## Etapa 1 - Importando as bibliotecas

In [ ]:
# instala uma versão específica do tensorflow, pra garantir compatibilidade com o resto do notebook
!pip install tensorflow==2.12.0

In [ ]:
import cv2
import numpy as np
import pandas as pd
from google.colab.patches import cv2_imshow
import zipfile

In [ ]:
# confere a versão do tensorflow instalada
import tensorflow
tensorflow.__version__

## Etapa 2 - Conectando com o Drive e acessando os arquivos

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# extrai o material do curso (haarcascade, fotos de teste etc) pra dentro do Colab
path = "/content/gdrive/My Drive/Material.zip"
zip_object = zipfile.ZipFile(file=path, mode="r")
zip_object.extractall("./")

In [ ]:
# extrai também o dataset fer2013, que fica zipado dentro do Material
base_imgs = 'Material/fer2013.zip'
zip_object = zipfile.ZipFile(file = base_imgs, mode = 'r')
zip_object.extractall('./')
zip_object.close

## Etapa 3 - Acessando a base com fotos de expressões faciais



In [ ]:
# carrega o csv com os dados: cada linha tem os pixels de uma imagem + o id da emoção
data = pd.read_csv('fer2013/fer2013.csv')
data.tail() # só pra ver as últimas linhas e conferir o formato

In [ ]:
# histograma pra ver como as emoções estão distribuídas no dataset (se tá balanceado ou não)
plt.figure(figsize=(12,6))
plt.hist(data['emotion'], bins = 30)
plt.title('Imagens x emoções')

# Classes: ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

## Etapa 4 - Pré-processamento

In [ ]:
# a coluna 'pixels' vem como uma string com os valores separados por espaço
pixels = data['pixels'].tolist()
pixels

In [ ]:
largura, altura = 48, 48
faces = []   # vai guardar todas as imagens já convertidas em matriz
amostras = 0
for pixel_sequence in pixels:
  # separa a string de pixels e converte cada valor pra inteiro
  face = [int(pixel) for pixel in pixel_sequence.split(' ')]
  # transforma a lista de pixels numa matriz 48x48 (formato de imagem)
  face = np.asarray(face).reshape(largura, altura)
  faces.append(face)

  if (amostras < 10): # mostra só as 10 primeiras, pra não travar o notebook
    cv2_imshow(face)
  amostras += 1

In [ ]:
# confere quantas imagens no total foram carregadas
print('Número total de imagens no dataset: ', str(len(faces)))

In [ ]:
# converte a lista de imagens para um array numpy
faces = np.asarray(faces)

In [ ]:
faces.shape

In [ ]:
# adiciona uma dimensão extra pro canal de cor (imagem em escala de cinza = 1 canal)
# necessário porque a camada Conv2D espera o formato (largura, altura, canais)
faces = np.expand_dims(faces, -1)
faces.shape

In [ ]:
# normaliza os pixels: de 0-255 para 0-1 (ajuda a rede a convergir mais rápido)
def normalizar(x):
  x = x.astype('float32')
  x = x / 255.0
  return x

In [ ]:
faces = normalizar(faces)

In [ ]:
faces[0]

In [ ]:
# transforma o id da emoção (0 a 6) em one-hot encoding, formato que a rede espera na saída
# ex: emoção 3 vira [0,0,0,1,0,0,0]
emocoes = pd.get_dummies(data['emotion']).values

In [ ]:
emocoes[0]

## Etapa 5 - Imports do Tensorflow/Keras

In [ ]:
from sklearn.model_selection import train_test_split # divide os dados em treino/teste/validação

# camadas e utilitários do keras pra montar a CNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# callbacks: controlam o treinamento (reduzir learning rate, parar cedo, salvar o melhor modelo)
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

from tensorflow.keras.models import load_model       # carregar um modelo já treinado (.h5)
from tensorflow.keras.models import model_from_json  # recriar a arquitetura a partir do json salvo

## Etapa 6 - Dividir em conjuntos para treinamento e validação

In [ ]:
# separa 10% dos dados pra teste
X_train, X_test, y_train, y_test = train_test_split(faces, emocoes, test_size = 0.1, random_state = 42)
# do que sobrou, separa mais 10% pra validação (usada durante o treinamento, pra acompanhar o overfitting)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.1, random_state = 41)

In [ ]:
print('Número de imagens no conjunto de treinamento:', len(X_train))
print('Número de imagens no conjunto de teste:', len(X_test))
print('Número de imagens no conjunto de validação:', len(X_val))

In [ ]:
# salva o conjunto de teste em disco, assim depois dá pra recarregar sem refazer o pré-processamento
np.save('mod_xtest', X_test)
np.save('mod_ytest', y_test)

## Etapa 7 - Arquitetura do Modelo (CNN)

### Arquitetura 1 do modelo

Padding same x valid: https://www.corvil.com/kb/what-is-the-difference-between-same-and-valid-padding-in-tf-nn-max-pool-of-tensorflow

Implementação original: https://medium.com/@birdortyedi_23820/deep-learning-lab-episode-3-fer2013-c38f2e052280

Regularizers: https://keras.io/regularizers/

Dropout: http://jmlr.org/papers/volume15/srivastava14a.old/srivastava14a.pdf


In [ ]:
# arquitetura 1: uma CNN "clássica" com 4 blocos convolucionais (cada um dobrando o número de filtros)
# e mais 3 camadas densas no final antes da saída
num_features = 64
num_labels = 7
batch_size = 64
epochs = 100
width, height = 48, 48

model = Sequential()

# bloco 1
model.add(Conv2D(num_features, kernel_size=(3,3), activation='relu',
                 input_shape=(width, height, 1), data_format = 'channels_last',
                 kernel_regularizer = l2(0.01))) # regularização L2 pra ajudar a evitar overfitting
model.add(Conv2D(num_features, kernel_size=(3,3), activation='relu', padding='same'))
model.add(BatchNormalization())              # normaliza as ativações, ajuda o treino a ser mais estável
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))  # reduz a dimensão espacial pela metade
model.add(Dropout(0.5))                      # "desliga" 50% dos neurônios aleatoriamente, evita overfitting

# bloco 2 (dobra o número de filtros)
model.add(Conv2D(2*num_features, kernel_size=(3,3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(Conv2D(2*num_features, kernel_size=(3,3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))
model.add(Dropout(0.5))

# bloco 3
model.add(Conv2D(2*2*num_features, kernel_size=(3,3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(Conv2D(2*2*num_features, kernel_size=(3,3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))
model.add(Dropout(0.5))

# bloco 4
model.add(Conv2D(2*2*2*num_features, kernel_size=(3,3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(Conv2D(2*2*2*num_features, kernel_size=(3,3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))
model.add(Dropout(0.5))

model.add(Flatten()) # achata o mapa de características em um vetor, pra entrar nas camadas densas

# camadas totalmente conectadas (classificador)
model.add(Dense(2*2*2*num_features, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(2*2*num_features, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(2*num_features, activation='relu'))
model.add(Dropout(0.5))

# camada de saída: 7 neurônios (uma pra cada emoção), softmax pra virar probabilidade
model.add(Dense(num_labels, activation = 'softmax'))

model.summary()

## Etapa 8 - Compilando o modelo

Parâmetros Adam: https://arxiv.org/abs/1412.6980

Artigo Adam: https://machinelearningmastery.com/adam-optimization-algorithm-for-deep-learning/

beta: Taxa de decaimento exponencial (por exemplo, 0.9)

In [ ]:
model.compile(loss = 'categorical_crossentropy',    # loss padrão pra classificação com várias classes
              optimizer = Adam(lr = 0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-7),
              metrics = ['accuracy'])

arquivo_modelo = 'modelo_01_expressoes.h5'
arquivo_modelo_json = 'modelo_01_expressoes.json'

# reduz o learning rate quando a loss de validação para de melhorar
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor = 0.9, patience=3, verbose = 1)
# para o treinamento mais cedo se não houver melhora (evita treinar epochs desnecessárias / overfitting)
early_stopper = EarlyStopping(monitor='val_loss', min_delta=0, patience = 8, verbose = 1, mode = 'auto')
# salva automaticamente o melhor modelo (menor val_loss) durante o treino
checkpointer = ModelCheckpoint(arquivo_modelo, monitor='val_loss', verbose = 1, save_best_only=True)

### Salvando a arquitetura do modelo em um arquivo JSON

In [ ]:
# salva só a arquitetura do modelo (as camadas) num arquivo json
# os pesos ficam separados, no .h5 salvo pelo checkpointer durante o treino
model_json = model.to_json()
with open(arquivo_modelo_json, 'w') as json_file:
  json_file.write(model_json)

## Etapa 9 - Treinando o modelo

In [ ]:
# treina o modelo
# history guarda o histórico de loss/acurácia de cada epoch, usamos depois pra plotar o gráfico
history = model.fit(np.array(X_train), np.array(y_train),
                    batch_size = batch_size,
                    epochs = epochs,
                    verbose = 1,
                    validation_data = (np.array(X_val), np.array(y_val)),
                    shuffle=True,
                    callbacks=[lr_reducer, early_stopper, checkpointer])

In [ ]:
print(history.history) # dicionário com os valores de loss/accuracy de cada epoch

## Gerando gráfico da melhora em cada etapa do treinamento

In [ ]:
# plota dois gráficos lado a lado: acurácia e loss (treino x validação) ao longo das epochs
def plota_historico_modelo(historico_modelo):
  fig, axs = plt.subplots(1, 2, figsize=(15,5))
  axs[0].plot(range(1, len(historico_modelo.history['accuracy']) + 1), historico_modelo.history['accuracy'], 'r')
  axs[0].plot(range(1, len(historico_modelo.history['val_accuracy']) + 1), historico_modelo.history['val_accuracy'], 'b')
  axs[0].set_title('Acurácia do modelo')
  axs[0].set_ylabel('Acurácia')
  axs[0].set_xlabel('Epoch')
  axs[0].set_xticks(np.arange(1, len(historico_modelo.history['accuracy']) + 1),
                    len(historico_modelo.history['accuracy']) / 10)
  axs[0].legend(['training accuracy', 'validation accuracy'], loc = 'best')

  axs[1].plot(range(1, len(historico_modelo.history['loss']) + 1), historico_modelo.history['loss'], 'r')
  axs[1].plot(range(1, len(historico_modelo.history['val_loss']) + 1), historico_modelo.history['val_loss'], 'b')
  axs[1].set_title('Loss do modelo')
  axs[1].set_ylabel('Loss')
  axs[1].set_xlabel('Epoch')
  axs[1].set_xticks(np.arange(1, len(historico_modelo.history['loss']) + 1),
                    len(historico_modelo.history['loss']) / 10)
  axs[1].legend(['training loss', 'validation loss'], loc = 'best')
  fig.savefig('historico_modelo_mod01.png') # salva a imagem do gráfico em disco

plota_historico_modelo(history)

### Verificando a acurácia do modelo

In [ ]:
# avalia o modelo já treinado usando o conjunto de teste (imagens que ele nunca viu)
scores = model.evaluate(np.array(X_test), np.array(y_test), batch_size = batch_size)

In [ ]:
scores

In [ ]:
print('Acurácia: ' + str(scores[1]))
print('Erro: ' + str(scores[0]))

## Carregamento dos dados para gerar a matriz de confusão

In [ ]:
true_y = []  # vai guardar o índice da emoção correta (rótulo verdadeiro)
pred_y = []  # vai guardar o índice da emoção que o modelo previu
# recarrega o conjunto de teste que salvamos antes
x = np.load('mod_xtest.npy')
y = np.load('mod_ytest.npy')

In [ ]:
x[0]

In [ ]:
y[0]

In [ ]:
# recarrega a arquitetura do modelo a partir do json salvo
json_file = open(arquivo_modelo_json, 'r')
loaded_model_json = json_file.read()
json_file.close()

In [ ]:
# recria o modelo com essa arquitetura e carrega os pesos treinados (.h5)
loaded_model = model_from_json(loaded_model_json)
loaded_model.load_weights(arquivo_modelo)

In [ ]:
# faz a previsão pra todo o conjunto de teste de uma vez
y_pred = loaded_model.predict(x)

In [ ]:
y_pred[0]

In [ ]:
yp = y_pred.tolist()
yt = y.tolist()
count = 0

In [ ]:
len(y)

In [ ]:
for i in range(len(y)):
  yy = max(yp[i]) # maior probabilidade prevista pelo modelo
  yyt = max(yt[i]) # valor "1" do one-hot (rótulo verdadeiro)
  pred_y.append(yp[i].index(yy))# índice da emoção prevista
  true_y.append(yt[i].index(yyt))# índice da emoção correta
  if (yp[i].index(yy) == yt[i].index(yyt)):
    count += 1 # conta quantas vezes o modelo acertou

acc = (count / len(y)) * 100

In [ ]:
print('Acurácia no conjunto de teste: ' + str(acc))

In [ ]:
# salva os resultados (usados depois pra montar a matriz de confusão)
np.save('truey_mod01', true_y)
np.save('predy_mod01', pred_y)

## Gerando a Matriz de Confusão

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
y_true = np.load('truey_mod01.npy')
y_pred = np.load('predy_mod01.npy')

In [ ]:
# monta a matriz comparando o rótulo real x o previsto
cm = confusion_matrix(y_true, y_pred)
expressoes = ['Raiva', 'Nojo', 'Medo', 'Feliz', 'Triste', 'Surpreso', 'Neutro']
titulo = 'Matriz de Confusão'
print(cm)

In [ ]:
# desenha a matriz de confusão como um "mapa de calor"
import itertools
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title(titulo)
plt.colorbar()
tick_marks = np.arange(len(expressoes))
plt.xticks(tick_marks, expressoes, rotation = 45)
plt.yticks(tick_marks, expressoes)
fmt = 'd'
thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
  # escreve o número dentro de cada célula da matriz
  plt.text(j, i, format(cm[i, j], fmt), horizontalalignment='center', color='white' if cm[i,j] > thresh else 'black')

plt.ylabel('Classificação correta')
plt.xlabel('Predição')
plt.savefig('matriz_confusao_mod01.png')

## Testando o modelo

In [ ]:
# carrega uma foto de teste, diferente das usadas no treinamento, pra testar o modelo
imagem = cv2.imread('Material/testes/teste02.jpg')
cv2_imshow(imagem)

In [ ]:
# faz uma cópia (pra não alterar a imagem original) e converte pra escala de cinza
original = imagem.copy()
gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
cv2_imshow(gray)

In [ ]:
# carrega o classificador haarcascade e detecta os rostos na imagem
face_cascade = cv2.CascadeClassifier('Material/haarcascade_frontalface_default.xml')
faces = face_cascade.detectMultiScale(gray, 1.1, 3)

In [ ]:
faces

In [ ]:
for (x, y, w, h) in faces:
  cv2.rectangle(original, (x, y), (x + w, y + h), (0, 255, 0), 1)  # desenha o retângulo em volta do rosto
  roi_gray = gray[y:y + h, x:x + w] # extrai só a região do rosto (ROI)
  roi_gray = roi_gray.astype('float') / 255.0 # normaliza
  cropped_img = np.expand_dims(np.expand_dims(cv2.resize(roi_gray, (48, 48)), -1), 0)  # redimensiona e ajusta o shape
  prediction = loaded_model.predict(cropped_img)[0] # prediz a emoção
  cv2.putText(original, expressoes[int(np.argmax(prediction))], (x, y - 10),
              cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)# escreve a emoção acima do rosto

cv2_imshow(original)